In [0]:
# Importando bibliotecas
from pyspark.sql.functions import col, lower, upper, initcap, month, year, when, count, regexp_replace
from pyspark.sql.types import IntegerType, DoubleType

In [0]:
# Recebendo Table Bronze
df_bronze = spark.table (
    "projeto_profissional_genz.genz.genz_social_media_bronze"
)

In [0]:
df_bronze.printSchema()

In [0]:
# Recebendo em tabela silver colunas selecionadas da tabela bronze
df_silver = df_bronze.select (
    "id",
    "data_inversa",
    "uf",
    "br",
    "km",
    "municipio",
    "causa_acidente",
    "tipo_acidente"
)

df_silver.show(10, truncate= False)

In [0]:
# Removendo Linhas em branco
df_silver = df_silver.dropna(
    subset=[
        "id",
        "data_inversa",
        "uf",
        "br",
        "km",
        "municipio",
        "causa_acidente"
    ]
)

df_silver.show(10, truncate=False)

In [0]:
# Removendo linhas duplicadas
df_silver = df_silver.drop_duplicates(
    subset=[
        "id",
        "data_inversa",
        "uf",
        "br",
        "km",
        "municipio",
        "causa_acidente"
    ]
)

df_silver.show(10, truncate=False)

In [0]:
# Renomeando coluna `data_inversa` para `data_acidente`
df_silver = df_silver.withColumnRenamed("data_inversa", "data_acidente")

df_silver.show(10, truncate=False)

In [0]:
# Padronização de valores das colunas
df_silver = df_silver.withColumn(
    "municipio", initcap(col("municipio"))
).withColumn(
    "causa_acidente", lower(col("causa_acidente"))
).withColumn(
    "tipo_acidente", lower(col("tipo_acidente"))
)

df_silver.show(5, truncate=False)

In [0]:
#Criando coluna `mês_acidente` e `ano_acidente`
df_silver = df_silver.withColumn (
    "mes_acidente", month(col("data_acidente"))
).withColumn (
    "ano_acidente", year(col("data_acidente"))
)

df_silver.show(5, truncate=False)

In [0]:
# Adicionando coluna `nome_mes`, com mes em string!
df_silver = df_silver.withColumn(
    "nome_mes",
    when(col("mes_acidente") == 1, "Janeiro")
    .when(col("mes_acidente") == 2, "Fevereiro")
    .when(col("mes_acidente") == 3, "Março")
    .when(col("mes_acidente") == 4, "Abril")
    .when(col("mes_acidente") == 5, "Maio")
    .when(col("mes_acidente") == 6, "Junho")
    .when(col("mes_acidente") == 7, "Julho")
    .when(col("mes_acidente") == 8, "Agosto")
    .when(col("mes_acidente") == 9, "Setembro")
    .when(col("mes_acidente") == 10, "Outubro")
    .when(col("mes_acidente") == 11, "Novembro")
    .when(col("mes_acidente") == 12, "Dezembro")
    .otherwise("Não informado")
)

df_silver.show(5, truncate=False)

In [0]:
#Contando NA em Coluna BR
df_silver.filter(col("br") == "NA").count()

In [0]:
#Contando NA em Coluna KM
df_silver.filter(col("km") == "NA").count()

In [0]:
#Tranformando NA em None
df_silver = df_silver.withColumn(
    "br",
     when(col("br") == "NA", None)
    .otherwise(col("br"))
)

df_silver = df_silver.withColumn(
    "km",
     when(col("km") == "NA", None)
    .otherwise(col("km"))
)

In [0]:
#Limpando Valores Nulos
df_silver = df_silver.dropna(
    subset=(["br"])
)

df_silver = df_silver.dropna(
    subset=(["km"])
)

In [0]:
#Transformando coluna KM de `,` para `.` e transformando em Double!
df_silver = df_silver.withColumn(
    "km",
    regexp_replace(col("km"), ",", ".").cast(DoubleType())
)

df_silver.printSchema()
df_silver.show(80, truncate=False)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

In [0]:
df = spark.table("projeto_profissional_bigdata.datatran_2018.silver_datatran18")
df.show()

In [0]:
%sql
select * from projeto_profissional_bigdata.datatran_2018.silver_datatran18